In [1]:
import pandas as pd

Overall goal: in building a predictive model, we want to successfully predict a hospital closure with enough lead time before it occurs to give the hospital sufficient leeway to make the necessary changes to stay open.

Reading in the full dataset, splitting out columns into categories, and dropping columns that are not needed to create the dataset I'll use for the model:

In [2]:
hospitals_full = pd.read_csv('../data/hospitals_full.csv')

/var/folders/51/zgq0lbb14t13h0_bgn8nntnm0000gn/T/ipykernel_17645/3201850437.py:1: DtypeWarning: Columns (67) have mixed types. Specify dtype option on import or set low_memory=False.
  hospitals_full = pd.read_csv('../data/hospitals_full.csv')


In [3]:
model_data = hospitals_full.drop(columns=['State', 'Zip', 'County','Facility Name','Closed', 'Closure Date','Full Address', 'Prior Name','Tract_Code', 'CBSA_Code', 'CBSA_Title','County_FIPS', 'State_FIPS'])


In [4]:
model_data.shape

(13391, 189)

In [5]:
model_data.head()

,Year,CCN,Medicaid charges,HAC reduction adjustment amount,STATEMENT OF REVENUES AND EXPENSES: Net Income (G3_C1_29),"ADJUSTED SALARIES, Subtotal Salaries",BALANCE SHEET: Prepaid expenses (G_C1THRU4_8),BED DAYS: Total Hospital,Number of Interns & Residents,BALANCE SHEET: Cash on Hand and in Banks (G_C1THRU4_1),...,"Per Capita Phys,Primary Care, Patient Care Non-Fed",Per Capita Short Term Gen Hosp Admissions,Per Capita Short Term General Hosp Beds,Per Capita Total Active D.O.s Non-Federal,Per Capita Total Active M.D.s Non-Federal,Per Capita Total Medicare Inpatient Days Short Term General Hospitals,Per Capita Total Number Hospitals,Percent Persons in Poverty,Population Estimate,"Unemployment Rate, 16+"
0,2010,190044,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.000340,0.092921,0.003529,0.0,0.000664,0.241335,0.000065,21.0,61773.0,6.8
1,2011,190044,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.000371,0.085783,0.003275,0.0,0.000710,0.230389,0.000065,22.4,61982.0,6.3
2,2012,190044,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.000355,0.083247,0.003279,0.0,0.000695,0.244573,0.000065,20.7,61912.0,5.6
3,2013,190044,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.000354,0.082776,0.003263,0.0,0.000675,0.262877,0.000064,19.0,62204.0,NaN
4,2014,190044,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.000368,0.082386,0.003249,0.0,0.000736,0.269148,0.000048,22.0,62486.0,5.7


#### Feature Engineering:

Creating "General Ownership" column to group hospital types into 'govt','non-profit', and 'for-profit':

In [6]:
model_data['Hospital Type'] = model_data['Hospital Type'].replace('Governmental, Federal', 'Government - Federal')

In [7]:
model_data['General Ownership Type'] = model_data['Hospital Type'].replace({'Voluntary non-profit - Other': 'non-profit',
                                            'Voluntary non-profit - Private': 'non-profit',
                                           'Voluntary non-profit - Church': 'non-profit',
                                           'Government - State': 'govt',
                                           'Government - Local': 'govt', 
                                           'Government - Hospital District or Authority': 'govt', 
                                           'Government - Federal': 'govt', 
                                           'Governmental, Other': 'govt',
                                           'Proprietary, Corporation': 'for-profit', 
                                           'Proprietary': 'for-profit', 
                                           'Tribal': 'govt',
                                           'Physician': 'for-profit',
                                           'Non profit - Corporation': 'non-profit'
                                          }, 
                                         )

In [8]:
model_data['General Ownership Type'].isna().sum()

np.int64(0)

In [9]:
model_data['Hospital Type'].nunique()

13

We might consider excluding hospital type from our model, since it will result in 13 unique features after encoding.

In [10]:
pd.set_option('display.max_rows', None)
model_data.isnull().mean().sort_values(ascending=False)

hf_readm_npatients                                                         1.000000
ami_readm_rate                                                             1.000000
hf_readm_rate                                                              1.000000
pn_mort_rate                                                               1.000000
pn_readm_rate                                                              1.000000
ami_mort_npatients                                                         1.000000
ami_readm_npatients                                                        1.000000
hf_mort_npatients                                                          1.000000
ami_mort_rate                                                              1.000000
pn_mort_npatients                                                          1.000000
pn_readm_npatients                                                         1.000000
all_readm_rate                                                             1

In [11]:
# Drop columns that are more than 80% NaNs
model_data = model_data.loc[:, model_data.isnull().mean() <= 0.80]

In [12]:
model_data.shape

(13391, 94)

This still leaves us with 91 features (94 columns minus 2 id columns and 1 target column)

Breaking down features into large scale types:

In [13]:
print(list(model_data.columns))

['Year', 'CCN', 'Medicaid charges', 'STATEMENT OF REVENUES AND EXPENSES: Net Income (G3_C1_29)', 'ADJUSTED SALARIES, Subtotal Salaries', 'BALANCE SHEET: Prepaid expenses (G_C1THRU4_8)', 'BED DAYS: Total Hospital', 'Number of Interns & Residents', 'BALANCE SHEET: Cash on Hand and in Banks (G_C1THRU4_1)', 'Cost To Charge Ratio', 'Total discharges', 'Cost of Uncompensated Care', 'BALANCE SHEET: Accounts Receivable (G_C1THRU4_4)', 'Total Salaries', 'BALANCE SHEET: Total Assets (G_C1THRU4_36)', 'BALANCE SHEET: Total Current Assets (G_C1THRU4_11)', 'REIMBURSEMENT SETTLEMENT: Interim payments', 'Total Bad Debt expense', 'TRIAL BALANCE OF EXPENSE ACCOUNTS: Interest Expense (A_C2_113)', 'NUMBER OF BEDS: Adults & Pediatrics', 'NUMBER OF BEDS: Total Hospital', 'Total Inpatient Days', 'Total Days Title XVIII', 'Total cost of charity care', 'Total Costs', 'NUMBER OF BEDS: ICU', 'Total Charges', 'Net Revenue from Medicaid', 'Total Liabilities', 'HVBP payment adjustment amount', 'REIMBURSEMENT SETTLE

In [14]:
id_cols = ['Year','CCN']
target_col = ['Closure Proximity']
hospital_char_cols = ['General Ownership Type','Emergency Services','Hospital Type']
financial_cols = ['BALANCE SHEET: Total Assets (G_C1THRU4_36)', 'BED DAYS: Total Hospital', 'REIMBURSEMENT SETTLEMENT: Interim payments', 
                  'IPPS Interim payment', 'BALANCE SHEET: Cash on Hand and in Banks (G_C1THRU4_1)', 
                  'BALANCE SHEET: Total Current Liabilities (G_C1THRU4_45)', 'Total cost of charity care', 
                  'HVBP payment adjustment amount', 'Total discharges', 'Total Salaries', 
                  'BALANCE SHEET: Accounts Receivable (G_C1THRU4_4)', 'Net Revenue from Medicaid', 
                  'Total Bad Debt expense', 'Total Charges', 'ADJUSTED SALARIES, Subtotal Salaries', 
                  'NUMBER OF BEDS: Adults & Pediatrics', 'Total Costs', 'BALANCE SHEET: Prepaid expenses (G_C1THRU4_8)', 
                  'BALANCE SHEET: Total Current Assets (G_C1THRU4_11)', 'Cost To Charge Ratio', 'REIMBURSEMENT SETTLEMENT: Subtotal', 
                  'Number of Interns & Residents', 'RECONCILIATION OF CAPITAL COST CENTERS: Depreciation, Total (A7_3_C9_3)', 
                  'Total Inpatient Days', 'STATEMENT OF REVENUES AND EXPENSES: Net Patient Revenue (G3_C1_3)', 
                  'NUMBER OF BEDS: Total Hospital', 'NUMBER OF BEDS: ICU', 'STATEMENT OF REVENUES AND EXPENSES: Net Income (G3_C1_29)', 
                  'BALANCE SHEET: Inventory (G_C1THRU4_7)', 'IPPS Payment amount (unadjusted)', 
                  'TRIAL BALANCE OF EXPENSE ACCOUNTS: Interest Expense (A_C2_113)', 'Cost of Uncompensated Care', 'Medicaid charges', 
                  'Total Liabilities', 'Total Days Title XVIII', 'Financial Indicators: Total Net Assets', 'S-10 DATA: Medicaid Costs', 
                  'Financial Indicators: Cash Reserves', 'Financial Indicators: Net Profit Margin', 'Financial Indicators: Operating Profit', 
                  'Financial Indicators: Operating Profit Margin', 'Financial Indicators: LIQUIDITY current ratio', 
                  'Financial Indicators: LIQUIDITY acid-test ratio', 'Financial Indicators: LIQUIDITY acid-test ratio (variation)', 
                  'Financial Indicators: LIQUIDITY cash ratio', 'Financial Indicators: SOLVENCY Debt-to-Equity Ratio', 
                  'Financial Indicators: SOLVENCY Debt Ratio', 'Financial Indicators: SOLVENCY Equity Ratio', 
                  'Financial Indicators: SOLVENCY Interest Coverage Ratio', 'Financial Indicators: SOLVENCY total assets less total liabilities', 
                  'Financial Indicators: EFFICIENCY asset turnover ratio', 'Financial Indicators: EFFICIENCY Accounts Receivable Turnover Ratio']
hosp_quality_cols = ['nsurveys', 'rrate', 'clean_score', 'commdoc_score', 'commnurse_score', 'explain_score', 'help_score', 
                     'info_score', 'overall_score', 'pain_score', 'quiet_score', 'recommend_score', 'understood_score']
demographic_cols = ['RUCA','Metro_Status','% <65 without Health Insurance', 'Dist Hosp By 00 - 39% Util Rate Short Term General Hospitals', 
                    'Dist Hosp By 40 - 59% Util Rate Short Term General Hospitals', 
                    'Dist Hosp By 60 - 79% Util Rate Short Term General Hospitals', 
                    'Dist Hosp By 80+% Util Rate Short Term General Hospitals', 
                    'Median Household Income', 'Per Capita # Short Term General Hosps', 'Per Capita Hospital Admissions', 
                    'Per Capita Hospital Beds', 'Per Capita Inpatient Days in ST Gen Hosp', 'Per Capita Personal Income', 
                    'Per Capita Phys,Primary Care, Patient Care Non-Fed', 'Per Capita Short Term Gen Hosp Admissions', 
                    'Per Capita Short Term General Hosp Beds', 'Per Capita Total Active D.O.s Non-Federal', 
                    'Per Capita Total Active M.D.s Non-Federal', 'Per Capita Total Medicare Inpatient Days Short Term General Hospitals', 
                    'Per Capita Total Number Hospitals', 'Percent Persons in Poverty', 'Population Estimate', 'Unemployment Rate, 16+']



In [15]:
len(id_cols) + len(target_col) + len(hospital_char_cols) + len(financial_cols) + len(hosp_quality_cols) + len(demographic_cols) == len(model_data.columns)


True

Are there any hospitals that we don't have any data for?

In [16]:
model_data[model_data[financial_cols + hosp_quality_cols + demographic_cols].isna().all(axis=1)]

,Year,CCN,Medicaid charges,STATEMENT OF REVENUES AND EXPENSES: Net Income (G3_C1_29),"ADJUSTED SALARIES, Subtotal Salaries",BALANCE SHEET: Prepaid expenses (G_C1THRU4_8),BED DAYS: Total Hospital,Number of Interns & Residents,BALANCE SHEET: Cash on Hand and in Banks (G_C1THRU4_1),Cost To Charge Ratio,...,Per Capita Short Term Gen Hosp Admissions,Per Capita Short Term General Hosp Beds,Per Capita Total Active D.O.s Non-Federal,Per Capita Total Active M.D.s Non-Federal,Per Capita Total Medicare Inpatient Days Short Term General Hospitals,Per Capita Total Number Hospitals,Percent Persons in Poverty,Population Estimate,"Unemployment Rate, 16+",General Ownership Type


For every hospital in the dataset, we have at least some financial data, quality data, and/or demographic data.

Ensuring all of the data types are appropriate for the values they represent:

In [17]:
model_data.dtypes

Year                                                                         int64
CCN                                                                          int64
Medicaid charges                                                           float64
STATEMENT OF REVENUES AND EXPENSES: Net Income (G3_C1_29)                  float64
ADJUSTED SALARIES, Subtotal Salaries                                       float64
BALANCE SHEET: Prepaid expenses (G_C1THRU4_8)                              float64
BED DAYS: Total Hospital                                                   float64
Number of Interns & Residents                                              float64
BALANCE SHEET: Cash on Hand and in Banks (G_C1THRU4_1)                     float64
Cost To Charge Ratio                                                       float64
Total discharges                                                           float64
Cost of Uncompensated Care                                                 float64
BALA

In [18]:
model_data['Closure Proximity'] = model_data['Closure Proximity'].astype(int)

In [19]:
model_data['RUCA'] = model_data['RUCA'].astype(object)

#### Data Preprocessing:

In [20]:
numeric_features = [col for col in model_data.select_dtypes('number').columns if col not in ['Year','CCN','Closure Proximity']]

In [21]:
categorical_features = [col for col in model_data.select_dtypes('object').columns]

In [22]:
model_data[categorical_features].isnull().mean().sort_values(ascending=False)

Emergency Services        0.0
RUCA                      0.0
Metro_Status              0.0
Hospital Type             0.0
General Ownership Type    0.0
dtype: float64

None of our categorical variables have missing values, so we don't need to worry about imputing them.

Next step: decide which type of imputer and scaler is best to use for the numeric features.

In [ ]:
# Numeric pipeline
numeric_transformer = Pipeline(steps=[
    # Applying an iterative imputer to fill missing values
    ("imputer", IterativeImputer(random_state=0, verbose=2)),
    # Applying a MaxAbsScaler (better than StandardScaler for sparse inputs or inputs that are not normally distributed)
    ("scaler", MaxAbsScaler())
])

# Categorical pipeline
categorical_transformer = Pipeline(steps=[
    ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
])

# Combine all the transformations into 1 preprocessor
# Using ColumnTransformer to select specific columns and apply transformations
preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ],
    remainder="passthrough",
    verbose_feature_names_out=False
)

pipe = Pipeline(
    steps=[("preprocessor", preprocessor)]
)

In [ ]:
# Train/test/validation split:


#### Hyperparameter Tuning:

Here, we'll split our data using 5-fold validation and apply the pre-processor pipeline (scaling and filling in missing values). 

Once we have our pre-processed cross-validation datasets, we'll fit a model to each of the training sets and evaluate its effectiveness via the corresponding validation set. 